# 🦴 FracAtlas Medical Vision-Language Model (VLM) Fine-Tuning
### Parameter-Efficient Fine-Tuning (QLoRA 4-bit) on Cloud GPU (Google Colab T4)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sahanawazhussain/medical-vlm-fracatlas/blob/main/notebooks/train_vlm.ipynb)

This notebook trains and specializes **Qwen2-VL** on the **FracAtlas** musculoskeletal radiograph dataset using **QLoRA 4-bit** quantization.

- **Target Hardware**: Google Colab Free Tier (NVIDIA Tesla T4 16GB GPU)
- **Base Model Options**: `Qwen/Qwen2-VL-2B-Instruct` (Fastest, ~1 hour) or `Qwen/Qwen2-VL-7B-Instruct` (~2.5 hours)
- **Output Artifact**: Trained LoRA adapter saved to `models/fracatlas_vlm_lora/` (~150 MB).

## 1. Google Colab Environment Setup & Repo Initialization

In [ ]:
import os, sys

# Verify GPU allocation
!nvidia-smi

# Auto-setup repository in Colab
if 'google.colab' in str(get_ipython()):
    print("\n[Colab] Initializing repository...")
    if not os.path.exists("medical-vlm-fracatlas"):
        !git clone https://github.com/Sahanawazhussain/medical-vlm-fracatlas.git
    %cd medical-vlm-fracatlas
    print("Current working directory:", os.getcwd())

# Install deep learning dependencies
!pip install -q --upgrade pip
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers accelerate peft bitsandbytes qwen-vl-utils pillow
print("\n✅ Environment dependencies successfully installed!")

## 2. Ingest Dataset & Run Multimodal Preprocessing
Downloads the 4,083 radiographs (if not present) and synthesizes multi-turn clinical QA pairs (`train.json`, `val.json`).

In [ ]:
# Download FracAtlas dataset (High-speed download inside Google cloud)
if not os.path.exists("data/raw/FracAtlas/FracAtlas/images"):
    print("Downloading FracAtlas radiographs...")
    !python download_dataset.py
else:
    print("Dataset already present.")

# Generate processed multimodal instruction pairs
if not os.path.exists("data/processed/train.json"):
    print("Running multimodal preprocessing...")
    !python src/dataset_to_vlm.py
else:
    print("Processed dataset ready.")

import json
with open("data/processed/train.json", "r") as f:
    train_data = json.load(f)
print(f"\nTotal Training Conversations Ready: {len(train_data)}")
print("Sample conversation:\n", json.dumps(train_data[0]['conversations'], indent=2))

## 3. Configure 4-bit Quantization (NF4) & Load Base VLM

In [ ]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Select target model: 'Qwen/Qwen2-VL-2B-Instruct' (Fast) or 'Qwen/Qwen2-VL-7B-Instruct'
MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"

print(f"Configuring 4-bit quantization for {MODEL_ID}...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
print("✅ Base model loaded in 4-bit precision!")

## 4. Inject LoRA Trainable Adapter Layers
Freezes 98.5% of base parameters and only trains lightweight attention projection matrices.

In [ ]:
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. Fine-Tuning Execution & Checkpointing

In [ ]:
from transformers import TrainingArguments, Trainer

OUTPUT_DIR = "models/fracatlas_vlm_lora"
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_ratio=0.03,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    report_to="none"
)

print("Saving fine-tuned LoRA adapter checkpoints...")
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"✅ LoRA adapter weights saved successfully to: {OUTPUT_DIR}")

## 6. Verification: Test Diagnostic Inference on Sample Radiograph

In [ ]:
# Run inference on a fractured test sample
!python src/inference.py --image data/raw/FracAtlas/FracAtlas/images/Fractured/IMG0000019.jpg

## 7. Run Quantitative Benchmark Evaluation (126 Test Images)

In [ ]:
# Generates confusion matrix, accuracy, sensitivity, and CSV results
!python src/evaluate.py

# Display summary results
with open("evaluation_results/benchmark_metrics_summary.json") as f:
    print(f.read())

## 8. Download Trained LoRA Adapter Zip
Exports the trained weights (~150 MB) so you can bring them to your local laptop or keep as project evidence.

In [ ]:
from google.colab import files
import shutil

# Zip the adapter directory
shutil.make_archive("fracatlas_vlm_lora", 'zip', "models/fracatlas_vlm_lora")
print("Created fracatlas_vlm_lora.zip")

# Trigger browser download
files.download("fracatlas_vlm_lora.zip")
print("Downloading adapter to your local machine!")